In [0]:
from pyspark import pipelines as dp

In [0]:
from pyspark.sql.functions import when,col,current_timestamp,date_diff,from_json,schema_of_json,explode,to_timestamp,to_date

In [0]:
@dp.table(
    name = "restaurant.silver.fact_order_items",
    table_properties= {"quality":"silver"}
)
@dp.expect_all_or_fail(
    {
      "validate_order_id":"order_id is not null",
      "validate_item_id" : "item_id is not null"
})
def silver_fact_order_items():
    df = spark.readStream.table("restaurant.bronze.historical_orders") \
            .withColumn("items",explode(from_json(col("items"),schema_of_json('[{"item_id": "ITEM-302", "name": "Chicken Tikka Masala", "category": "Main Course", "quantity": 2, "unit_price": 52.43, "subtotal": 104.86}]')))) \
            .withColumn("item_id",col("items.item_id")) \
            .withColumn("order_timestamp",to_timestamp("timestamp")) \
            .withColumn("order_date",to_date("timestamp")) \
            .withColumn("item_name",col("items.name")) \
            .withColumn("category",col("items.category")) \
            .withColumn("quantity",col("items.quantity")) \
            .withColumn("unit_price",col("items.unit_price")) \
            .withColumn("subtotal",col("items.subtotal")) \
            .withColumn("_ingestion_timestamp",current_timestamp()) \
            .select("order_id","item_id","restaurant_id","order_timestamp","order_date","item_name","category","quantity","unit_price","subtotal","_ingestion_timestamp")
    return df